# Minimal ShinkaEvolve Example, in One Notebook

This is the notebook version of `Examples_Shinkaevolve/1a_minimal_example`, but it is self-contained: the starter program, evaluator, prompt, and Shinka configuration are all written from cells below.

The task is intentionally tiny. We ask Shinka to improve a Python search routine for a two-variable objective, then inspect the resulting program database from the notebook.

What this notebook does:

- creates a notebook-owned task folder
- writes `initial.py`, `evaluate.py`, `prompt.txt`, and `shinka.yaml`
- smoke-tests the starter and evaluator without shelling out
- optionally runs Shinka through the Python API
- loads the resulting candidates into a dataframe for inspection


In [ ]:
from __future__ import annotations

import importlib.util
import math
import os
from pathlib import Path
from types import ModuleType

try:
    from IPython.display import Markdown, display
except ModuleNotFoundError:
    class Markdown(str):
        pass

    def display(value):
        print(value)


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / "pyproject.toml").exists() and (path / "Examples_Shinkaevolve").exists():
            return path
    raise RuntimeError("Could not find the Shinka_Yale repo root.")


repo_root = find_repo_root()
notebook_dir = repo_root / "Examples_Shinkaevolve" / "1b_minimal_example_notebook"
task_dir = notebook_dir / "minimal_example_task"
task_dir.mkdir(parents=True, exist_ok=True)

def load_repo_env() -> None:
    for env_path in [repo_root / ".env.local", repo_root / ".env"]:
        if not env_path.exists():
            continue
        try:
            from dotenv import load_dotenv

            load_dotenv(env_path, override=False)
        except ModuleNotFoundError:
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                os.environ.setdefault(key.strip(), value.strip().strip("'\""))
        print(f"loaded {env_path.relative_to(repo_root)}")
        return


load_repo_env()
print(f"repo_root = {repo_root}")
print(f"task_dir  = {task_dir}")


## The Optimization Problem

The fixed objective is:

```python
f(x, y) = sin(x) * cos(y) + sin(x * y) + (x**2 + y**2) / 20
```

The starter implementation returns the initial guess, so its score is deliberately boring. Shinka is only allowed to edit code inside the evolve block in `initial.py`; the evaluator checks that proposed programs return valid `(x, y, value)` triples inside the bounds.


In [ ]:
INITIAL_PY = r"""
import math
# You may import additional standard libraries if needed,
# such as random, statistics, numpy, scipy, etc.

# EVOLVE-BLOCK-START

def search_algorithm(
    seed: int = 0,
    initial_guess: tuple[float, float] = (0.0, 0.0),
    bounds: tuple[float, float] = (-5.0, 5.0),
):
    # Simplest possible baseline: return the initial guess.
    # The algorithm must be deterministic for the same seed and parameters.
    x, y = initial_guess
    return x, y, objective(x, y)


# EVOLVE-BLOCK-END

# The code outside of the evolve block is immutable scaffolding.

def objective(x: float, y: float) -> float:
    return float(math.sin(x) * math.cos(y) + math.sin(x * y) + (x**2 + y**2) / 20.0)


def run_search(
    seed: int = 0,
    iterations: int = 250,
    bounds: tuple[float, float] = (-5.0, 5.0),
):
    return search_algorithm(seed=seed, bounds=bounds)


if __name__ == "__main__":
    x, y, value = run_search()
    print(f"x={x:.6f} y={y:.6f} value={value:.6f}")
""".strip() + "\n"

PROMPT_TXT = r"""
You are improving a tiny optimization program.

The fixed objective is:
f(x, y) = sin(x) * cos(y) + sin(x * y) + (x**2 + y**2) / 20

The evaluator calls run_search(seed=..., iterations=..., bounds=...).
Improve search_algorithm so it finds lower values more reliably across seeds.

Keep the code short, simple, and fast.

Good ideas:
- better exploration than plain random search
- local refinement around good points
- multiple restarts
- adaptive step sizes
""".strip() + "\n"

print("Defined the starter program and task prompt in notebook variables.")


In [ ]:
EVALUATE_PY = r"""
#!/usr/bin/env python3
import argparse
import math
import os

from shinka.core import run_shinka_eval


BOUNDS = (-5.0, 5.0)
SEEDS = [0, 1, 42]

# This example is didactic, so we can include a rough reference optimum.
GLOBAL_MIN_X = -1.704
GLOBAL_MIN_Y = 0.678
GLOBAL_MIN_VALUE = -1.519


def objective(x: float, y: float) -> float:
    return float(math.sin(x) * math.cos(y) + math.sin(x * y) + (x**2 + y**2) / 20.0)


def validate_search_result(run_output, atol: float = 1e-6):
    try:
        x, y, value = (float(v) for v in run_output)
    except (TypeError, ValueError):
        return False, "run_search must return (x, y, value)."

    if not all(math.isfinite(v) for v in (x, y, value)):
        return False, "run_search returned non-finite values."

    low, high = BOUNDS
    if not (low <= x <= high and low <= y <= high):
        return False, "Returned point is outside the search bounds."

    if abs(objective(x, y) - value) > atol:
        return False, "Reported value does not match the objective."

    return True, None


def aggregate_search_metrics(results: list[tuple[float, float, float]]) -> dict:
    if not results:
        return {"combined_score": 0.0}

    xs = [float(x) for x, _, _ in results]
    ys = [float(y) for _, y, _ in results]
    values = [float(value) for _, _, value in results]

    mean_value = sum(values) / len(values)
    value_score = -mean_value  # ShinkaEvolve maximizes scores.

    mean_distance = sum(
        math.sqrt((x - GLOBAL_MIN_X) ** 2 + (y - GLOBAL_MIN_Y) ** 2)
        for x, y in zip(xs, ys)
    ) / len(results)
    distance_score = -mean_distance

    reliability_score = len(results) / len(SEEDS)
    combined_score = value_score + 0.3 * distance_score + 0.3 * reliability_score

    return {
        "combined_score": combined_score,
        "public": {
            "mean_value": mean_value,
            "mean_distance_to_known_minimum": mean_distance,
            "successful_runs": len(results),
        },
        "private": {
            "value_score": value_score,
            "distance_score": distance_score,
            "reliability_score": reliability_score,
        },
    }


def get_experiment_kwargs(run_index: int) -> dict:
    return {"seed": SEEDS[run_index], "bounds": BOUNDS}


def main(program_path: str, results_dir: str):
    os.makedirs(results_dir, exist_ok=True)

    metrics, correct, error_msg = run_shinka_eval(
        program_path=program_path,
        results_dir=results_dir,
        experiment_fn_name="run_search",
        num_runs=len(SEEDS),
        run_workers=1,
        get_experiment_kwargs=get_experiment_kwargs,
        validate_fn=validate_search_result,
        aggregate_metrics_fn=aggregate_search_metrics,
    )

    print(f"Evaluated program: {program_path}")
    print(f"Results saved to: {results_dir}")
    print(f"Correct: {correct}")
    if error_msg:
        print(f"Error: {error_msg}")
    print(f"Combined score: {metrics.get('combined_score', 0.0):.6f}")
    return metrics, correct, error_msg


if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Evaluate the minimal two-variable optimization example"
    )
    parser.add_argument("--program_path", type=str, default="initial.py")
    parser.add_argument("--results_dir", type=str, default="results")
    args = parser.parse_args()
    main(args.program_path, args.results_dir)
""".strip() + "\n"

SHINKA_YAML = r"""
max_evaluation_jobs: 2
max_proposal_jobs: 1
max_db_workers: 2
verbose: true

job:
  eval_program_path: evaluate.py
  time: "00:02:00"

db:
  db_path: evolution_db.sqlite
  num_islands: 1
  archive_size: 12
  elite_selection_ratio: 0.25
  num_archive_inspirations: 1
  num_top_k_inspirations: 1

evo:
  init_program_path: initial.py
  results_dir: results
  language: python
  job_type: local
  num_generations: 20
  max_api_costs: 4.0
  patch_types: [diff, full]
  patch_type_probs: [0.8, 0.2]
  max_patch_resamples: 1
  max_patch_attempts: 1
  llm_models: [openrouter/google/gemini-3-flash-preview]
  llm_kwargs:
    temperatures: [0.0, 0.7]
    reasoning_efforts: [low]
    max_tokens: 8192
  embedding_model: null
""".strip() + "\n"

print("Defined the evaluator and Shinka config in notebook variables.")


## Write the Task Files

Shinka expects real files so it can patch programs and run the evaluator in isolated jobs. The notebook still owns those files: re-running this cell recreates the task folder from the cell contents above.


In [ ]:
task_files = {
    "initial.py": INITIAL_PY,
    "evaluate.py": EVALUATE_PY,
    "prompt.txt": PROMPT_TXT,
    "shinka.yaml": SHINKA_YAML,
}

for filename, contents in task_files.items():
    path = task_dir / filename
    path.write_text(contents)
    print(f"wrote {path.relative_to(repo_root)}")


## Inspect What We Wrote

This is the whole minimal example on disk. There is no hidden copy step from another folder.


In [ ]:
def display_file(path: Path) -> None:
    suffix_to_language = {".py": "python", ".yaml": "yaml", ".txt": "text"}
    language = suffix_to_language.get(path.suffix, "text")
    display(Markdown(f"### `{path.name}`\n```{language}\n{path.read_text()}\n```"))


for filename in ["prompt.txt", "initial.py", "evaluate.py", "shinka.yaml"]:
    display_file(task_dir / filename)


## Smoke-Test the Starter Program

First we import the generated `initial.py` directly and call `run_search`. The baseline should return `(0, 0, 0)`.


In [ ]:
def import_module_from_path(name: str, path: Path) -> ModuleType:
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Could not import {path}")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


initial = import_module_from_path("minimal_initial", task_dir / "initial.py")
x, y, value = initial.run_search(seed=0)
print(f"x={x:.6f} y={y:.6f} value={value:.6f}")


## Smoke-Test the Evaluator

Now we import the generated evaluator and run it against the starter program. This uses `run_shinka_eval` directly, so failures show up as Python errors in the notebook instead of being hidden behind a shell command.


In [ ]:
evaluate = import_module_from_path("minimal_evaluate", task_dir / "evaluate.py")
smoke_dir = task_dir / "smoke_test"
metrics, correct, error_msg = evaluate.main(
    program_path=str(task_dir / "initial.py"),
    results_dir=str(smoke_dir),
)

print({"correct": correct, "error_msg": error_msg, **metrics})


## Run Shinka

This cell launches the evolutionary run through Shinka's Python API. It uses `shinka.yaml` for the run settings and `prompt.txt` for the system prompt.

Set `RUN_EVOLUTION = True` when your API key is configured. In this repo the default model goes through OpenRouter, so `OPENROUTER_API_KEY` should be present in the environment before running the cell.


In [ ]:
RUN_EVOLUTION = False

if not RUN_EVOLUTION:
    print("Set RUN_EVOLUTION = True to launch Shinka from the notebook.")
elif not os.getenv("OPENROUTER_API_KEY"):
    print("OPENROUTER_API_KEY is missing.")
    print("You should have received an email containing the full command to copy-paste:")
    print("echo 'OPENROUTER_API_KEY=...' >> .env.local")
    print("Run that command from the repository root, then verify with bash scripts/doctor.sh.")
else:
    import yaml
    from shinka.core import EvolutionConfig, ShinkaEvolveRunner
    from shinka.database import DatabaseConfig
    from shinka.launch import LocalJobConfig

    cfg = yaml.safe_load((task_dir / "shinka.yaml").read_text())
    cfg["evo"]["task_sys_msg"] = (task_dir / "prompt.txt").read_text().strip()

    old_cwd = Path.cwd()
    os.chdir(task_dir)
    try:
        runner = ShinkaEvolveRunner(
            evo_config=EvolutionConfig(**cfg["evo"]),
            job_config=LocalJobConfig(**cfg["job"]),
            db_config=DatabaseConfig(**cfg["db"]),
            max_evaluation_jobs=cfg["max_evaluation_jobs"],
            max_proposal_jobs=cfg["max_proposal_jobs"],
            max_db_workers=cfg["max_db_workers"],
            verbose=cfg.get("verbose", True),
        )
        runner.run()
    finally:
        os.chdir(old_cwd)


## Inspect Results

After evolution runs, this cell loads the candidate database and shows the strongest programs. The results live inside the notebook task folder, which keeps them visible to the repo's WebUI workflow.


In [ ]:
results_dir = task_dir / "results"
db_path = results_dir / "evolution_db.sqlite"

if not db_path.exists():
    print(f"No Shinka database yet at {db_path.relative_to(repo_root)}")
    print("Run the evolution cell first, or open the task folder in the WebUI after a run.")
else:
    from shinka.utils import load_programs_to_df

    df = load_programs_to_df(str(db_path))
    score_columns = [
        col
        for col in ["combined_score", "mean_value", "mean_distance_to_known_minimum"]
        if col in df.columns
    ]
    display(
        df.sort_values("combined_score", ascending=False)
        .head(10)[score_columns + ["program_id", "generation"]]
    )


## Load the Best Program

This final cell imports `results/best/main.py` and evaluates it on the same seeds used by the scorer, so you can sanity-check what Shinka discovered without leaving the notebook.


In [ ]:
best_program = results_dir / "best" / "main.py"

if not best_program.exists():
    print(f"No best program yet at {best_program.relative_to(repo_root)}")
else:
    best = import_module_from_path("minimal_best", best_program)
    rows = []
    for seed in evaluate.SEEDS:
        bx, by, bvalue = best.run_search(seed=seed, bounds=evaluate.BOUNDS)
        rows.append({"seed": seed, "x": bx, "y": by, "value": bvalue})
    display(rows)
